In [ ]:
# Crawl outputs of open-gira TC/grid analysis, specifically:
# power/by_storm_set/{STORM_SET}/disruption/pop_affected_RP/{RETURN_PERIOD}.gpq files for the future cases
# Plot line plot of global population disrupted for each GCM, normalised by GCM-mean

In [ ]:
from collections import defaultdict
from pathlib import Path
import glob
import os
import re

import numpy as np
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import matplotlib
import pandas as pd

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = Path("../plots")

# Do not extrapolate beyond the available events
tc_model_n_years = {
    "STORM": 1000,
    "CHAZ": 1000,
    "emanuel": 200
}

# Mapping from open-gira STORM_SET to display name
atmospheres_to_scenario = {
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM-CM6-1-HR",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC-CM2-VHR4",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth3P-HR",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3-GC31-HM",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM-CM6-1",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1-0-LL",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM-CM6-1",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth3",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM-CM6-1-HR",
        "2050 STORM RCP8.5 CMCC-CM2-VHR4",
        "2050 STORM RCP8.5 EC-Earth3P-HR",
        "2050 STORM RCP8.5 HadGEM3-GC31-HM"
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM-CM6-1",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
        "2050 CHAZ SSP5-8.5 UKESM1-0-LL",
    ],   
    "MIT SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM-CM6-1",
        "2050 Emanuel EC-Earth3",
        "2050 Emanuel MIROC6",
        "2050 Emanuel UKMO6",
    ],
}

# Equilibrium Climate Sensitivity (ECS) for each of the GCMs
ecs_data = [
  {
    "gcm_name": "CNRM-CM6-1-HR",
    "ecs_kelvin": 4.33,
    "source": "Zelinka et al. (2020), Causes of Higher Climate Sensitivity in CMIP6 Models, GRL — CMIP6_ECS_ERF_fbks.txt (github.com/mzelinka/cmip56_forcing_feedback_ecs), variant r1i1p1f2",
    "notes": "Direct value, not gap-filled."
  },
  {
    "gcm_name": "CMCC-CM2-VHR4",
    "ecs_kelvin": 3.55,
    "source": "Gap-filled from sister model CMCC-CM2-SR5 (same CMCC-CM2 family, standard CMIP6 resolution) — Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f1",
    "notes": "CMCC-CM2-VHR4 is a HighResMIP-only configuration that did not run the coupled abrupt-4xCO2 experiment, so no standalone ECS is published. CMCC-CM2-SR5 is the closest documented relative (same base model, coarser resolution), used here as a proxy. Actual VHR4 value could differ since resolution changes can shift cloud feedbacks."
  },
  {
    "gcm_name": "EC-Earth3P-HR",
    "ecs_kelvin": 4.26,
    "source": "Gap-filled from sister model EC-Earth3 (standard CMIP6 flagship resolution) — Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r8i1p1f1",
    "notes": "EC-Earth3P-HR is the HighResMIP high-resolution configuration of the same base model as EC-Earth3; it did not run abrupt-4xCO2 standalone. A 2026 Scientific Data paper (Yakubu et al.) reports its 4-model HighResMIP set spans ECS 2.99-5.62°C collectively but gives no per-model breakdown in the text I could access. Haarsma et al. (2020) note the HighResMIP protocol applies no extra tuning at high resolution, supporting EC-Earth3 as a reasonable proxy, though not a guarantee of an identical value."
  },
  {
    "gcm_name": "HadGEM3-GC31-HM",
    "ecs_kelvin": 5.44,
    "source": "Gap-filled from sister model HadGEM3-GC31-MM (same model family, closest resolution sibling) — Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt",
    "notes": "HadGEM3-GC31-HM (high-res atmosphere/medium-res ocean) has no published standalone abrupt-4xCO2 ECS. MM (medium atmosphere/medium ocean) is the closest documented sibling and was used as the proxy; LL (low-res) is also available at ECS = 5.55 K if a lower-resolution match is preferred. Andrews et al. (2019, JAMES) corroborates ECS ≈ 5.4-5.5 K across the HadGEM3-GC3.1 family."
  },
  {
    "gcm_name": "UKESM1-0-LL",
    "ecs_kelvin": 5.36,
    "source": "Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f2",
    "notes": "Direct value, not gap-filled. Corroborated by Andrews et al. (2019, JAMES), which reports ECS ≈ 5.4 K for UKESM1."
  },
  {
    "gcm_name": "CESM2",
    "ecs_kelvin": 5.15,
    "source": "Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f1",
    "notes": "Direct value, not gap-filled."
  },
  {
    "gcm_name": "CNRM-CM6-1",
    "ecs_kelvin": 4.90,
    "source": "Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f2",
    "notes": "Direct value, not gap-filled."
  },
  {
    "gcm_name": "EC-Earth3",
    "ecs_kelvin": 4.26,
    "source": "Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r8i1p1f1",
    "notes": "Direct value, not gap-filled."
  },
  {
    "gcm_name": "MIROC6",
    "ecs_kelvin": 2.60,
    "source": "Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f1",
    "notes": "Direct value, not gap-filled."
  },
  {
    "gcm_name": "UKMO6",
    "ecs_kelvin": 5.36,
    "source": "Gap-filled/assumed equal to UKESM1-0-LL — Zelinka et al. (2020) CMIP6_ECS_ERF_fbks.txt, variant r1i1p1f2",
    "notes": "The MIT/emanuel dict's short codes (cnrm6, ecearth6, ukmo6, plus explicit cesm2/miroc6) line up almost one-to-one with the CHAZ list's GCM set (CNRM-CM6-1, EC-Earth3, UKESM1-0-LL, CESM2, MIROC6), so 'ukmo6' is most likely UKESM1-0-LL. The alternative candidate is HadGEM3-GC31-LL (ECS = 5.55 K), the other UK Met Office CMIP6 flagship. Worth confirming against the actual model documentation/citation used to build that dict rather than treating this as certain."
  }
]
ecs_data = pd.DataFrame(ecs_data).set_index("gcm_name")
ecs_data

In [ ]:
# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)

#rps = [i * base for base in (1, 10, 100) for i in range(1, 10)]
rps = np.logspace(0, 3, 40)
baseline = {}
future = {}
for atmosphere, scenario in atmospheres_to_scenario.items():
    
    print(atmosphere)
    paths = glob.glob(os.path.join(results_dir, f"power/by_country/*/disruption/{atmosphere}/pop_affected_by_event.pq"))
    data = {path.split("/")[-4]: pd.read_parquet(path,) for path in paths}
    calibrated_data = {}
    for iso_a3, df in data.items():
        try:
            threshold = thresholds.loc[iso_a3, "threshold_ms-1"]
            calibrated_data[iso_a3] = df.loc[:, str(threshold)]
        except KeyError:
            pass
            #print(f"{iso_a3} missing...")
            
    concat = pd.concat(calibrated_data.values())
    concat = pd.DataFrame(concat).rename(columns={0: "pop_affected"})
    
    global_sum = concat.groupby(concat.index).sum()
    global_sum = global_sum.reset_index()
    if atmosphere.startswith("STORM") or atmosphere.startswith("IRIS"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(s.split("_")[2]))
    elif atmosphere.startswith("CHAZ") or atmosphere.startswith("emanuel"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(re.search(r"Y(\d{4})", s).groups()[0]))
        
    annual_max = global_sum.drop(columns=["event_id"]).groupby("year").max()
    annual_sum = global_sum.drop(columns=["event_id"]).groupby("year").sum()
    
    df = annual_sum.sort_values("pop_affected")
    df = df.reset_index()
    
    # For any atmospheres with more than 1 millenium, drop extra years
    df = df[(df.year >= 0) & (df.year < 1000)]

    # Label with plotting position
    n_years = df.year.max() - df.year.min() + 1
    year_range = range(n_years - 1, -1, -1)
    df["rank"] = year_range
    df["return_period_years"] = (n_years + 1) / df["rank"]

    # Interpolate to common RP grid
    df = df.drop(columns=["year", "rank"])
    df = df[np.isfinite(df.return_period_years)]
    df["log_rp"] = np.log(df.return_period_years)
    target = pd.DataFrame({"return_period_years": rps, "log_rp": np.log(rps), "pop_affected": np.nan})
    df = pd.concat([df, target]).sort_values("log_rp").set_index("log_rp")
    df = df.interpolate(method="index")
    df = df.loc[np.log(rps)].reset_index(drop=True)

    years_of_events = tc_model_n_years[atmosphere.split("_")[0]]
    df.loc[df.return_period_years > years_of_events, "pop_affected"] = np.nan

    for group_name, members in gcm_groups.items():
        if scenario in members:
            break

    family, *other = group_name.split()
    if other[0] == "baseline":
        baseline[(family, scenario)] = df
    else:
        future[(family, scenario)] = df

In [ ]:
f, ax = plt.subplots(figsize=(6,4))
data = defaultdict(list)
for (tc_family, scenario), df in future.items():
    gcm = scenario.split()[-1]
    data[tc_family].append(df.set_index("return_period_years").rename(columns={"pop_affected": scenario}))
    
for tc_family in data.keys():
    df = pd.concat(data[tc_family], axis=1)
    df = df.div(df.mean(axis=1), axis=0)
    old_index = df.columns.to_frame()
    old_index.insert(0, 'TC model', [tc_family] * len(old_index))
    df.columns = pd.MultiIndex.from_frame(old_index)
    data[tc_family] = df

data = pd.concat(data.values(), axis=1)

tc_model_ls = {
    "CHAZ": "-.",
    "IRIS": "--",
    "MIT": ":",
    "STORM": "-",
}

cmap = plt.get_cmap("YlOrRd")
norm = matplotlib.colors.Normalize(vmin=ecs_data.ecs_kelvin.min(), vmax=ecs_data.ecs_kelvin.max())

for tc_model, scenario in data.columns:
    ecs = ecs_data.loc[scenario.split()[-1], "ecs_kelvin"]
    ax.plot(data.index, data[(tc_model, scenario)], ls=tc_model_ls[tc_model], color=cmap(norm(ecs)))
ax.set_xscale("log")
ax.set_xlabel("Return period [years]", labelpad=8)
ax.set_ylabel("Global pop. affected, normalised by\nmean of TC model GCM members", labelpad=10)
ax.grid(which="both", alpha=0.2)
ax.set_ylim(0.62, 1.35)

cax = f.add_axes([0.93, 0.11, 0.03, 0.77])
sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = f.colorbar(sm, cax=cax)
cbar.set_label('Input GCM ECS [Kelvin]', labelpad=15)

# Create custom legend handles for TC models
legend_handles = [
    Line2D([0], [0], color='black', linestyle=ls, label=tc_model)
    for tc_model, ls in tc_model_ls.items()
]

# Add legend to axis
ax.legend(handles=legend_handles, loc="upper center", ncol=4,)

f.savefig(plot_dir / "pop_at_risk_RP_GCM_ECS.pdf", bbox_inches="tight")